In [1]:
!pip install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.6 MB/s eta 0:00:00


# Instalação

In [2]:
from ultralytics import YOLO
import cv2
import random

# Modelo
model = YOLO("yolo11n.pt")

# Cores por classe
class_colors = {}

def get_color(class_id):
    if class_id not in class_colors:
        class_colors[class_id] = (
            random.randint(0,255),
            random.randint(0,255),
            random.randint(0,255)
        )
    return class_colors[class_id]

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# Detecção com filtro de classes

In [3]:
def detect_objects(frame, model, classes=None, conf=0.25):
    """
    classes: lista de IDs (ex: [0,2,3]) ou None para todas
    """
    results = model.predict(frame, conf=conf, classes=classes, imgsz=640)
    return results

# Contador de objetos

In [4]:
def count_objects(results, model):
    counts = {}

    for result in results:
        for box in result.boxes:
            cls = int(box.cls[0])
            label = model.names[cls]

            counts[label] = counts.get(label, 0) + 1

    return counts

# Tracking com ID

In [5]:
def track_objects(frame, model, classes=None, conf=0.25):
    """
    Retorna objetos com ID persistente + filtro de classes
    """
    results = model.track(
        frame,
        persist=True,
        classes=classes,
        conf=conf,
        imgsz=640
    )
    return results

# Desenhar boxes + labels + IDs

In [6]:
def draw_boxes(frame, results, model, show_id=False):
    h, w, _ = frame.shape

    #  Escala dinâmica baseada na resolução
    font_scale = max(0.6, min(w, h) / 800)
    thickness = max(1, int(font_scale * 2))

    for result in results:
        boxes = result.boxes

        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            cls = int(box.cls[0])

            label = model.names[cls]
            color = get_color(cls)

            # ID (tracking)
            track_id = None
            if show_id and hasattr(box, "id") and box.id is not None:
                track_id = int(box.id[0])

            text = f"{label} {conf:.2f}"
            if track_id is not None:
                text += f" ID:{track_id}"

            # BOX mais grossa
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness + 1)

            # Tamanho do texto
            (tw, th), _ = cv2.getTextSize(
                text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness
            )

            # Fundo maior com padding
            padding = 6
            cv2.rectangle(
                frame,
                (x1, y1 - th - padding*2),
                (x1 + tw + padding*2, y1),
                color,
                -1
            )

            # Texto maior e centralizado
            cv2.putText(
                frame,
                text,
                (x1 + padding, y1 - padding),
                cv2.FONT_HERSHEY_SIMPLEX,
                font_scale,
                (255, 255, 255),
                thickness,
                cv2.LINE_AA
            )

    return frame

# Segmentação (SAM)

In [7]:
from ultralytics import SAM

segmenter = SAM("sam2.1_b.pt")

def segment_objects(frame, results):
    bboxes = []

    for result in results:
        for box in result.boxes.xyxy:
            bboxes.append(box.tolist())

    if not bboxes:
        return frame

    seg_results = segmenter.predict(frame, bboxes=bboxes, imgsz=384)

    for r in seg_results:
      masks = r.masks
      if masks is not None:
          for mask in masks.data:
              mask = mask.cpu().numpy()
              frame[mask > 0.5] = frame[mask > 0.5] * 0.5 + 100

    return frame

# Pipeline principal

In [8]:
def run_pipeline(
    input_path,
    output_path,
    use_tracking=False,
    use_segmentation=False,
    classes=None
):
    cap = cv2.VideoCapture(input_path)

    is_video = cap.isOpened() and int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) > 1

    if is_video:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    else:
        frame = cv2.imread(input_path)

    while True:
        if is_video:
            ret, frame = cap.read()
            if not ret:
                break

        # DETECÇÃO / TRACKING
        if use_tracking:
            results = track_objects(frame, model, classes=classes)
        else:
            results = detect_objects(frame, model, classes=classes)

        # CONTAGEM
        counts = count_objects(results, model)
        print("Contagem:", counts)

        # SEGMENTAÇÃO
        if use_segmentation:
            frame = segment_objects(frame, results)

        # DESENHO
        frame = draw_boxes(frame, results, model, show_id=use_tracking)

        if is_video:
            out.write(frame)
        else:
            cv2.imwrite(output_path, frame)
            break

    if is_video:
        cap.release()
        out.release()

    print("✅ Finalizado:", output_path)

# Executar

In [13]:
run_pipeline(
    input_path="/content/bananaa.jpeg",
    output_path="/content/bananaa_output.jpeg",
    classes=[0,16, 46], # 0=person, 16=dog, 46=banana
    use_tracking=True,
    use_segmentation=True
)


0: 640x640 1 banana, 11.6ms
Speed: 3.2ms preprocess, 11.6ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 640)
Contagem: {'banana': 1}

0: 384x384 1 0, 66.7ms
Speed: 1.8ms preprocess, 66.7ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 384)
✅ Finalizado: /content/bananaa_output.jpeg
